# P8-4 · Backend và dữ liệu cho AI

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

## Lý thuyết

In [ ]:
import hashlib, json, fakeredis
r = fakeredis.FakeRedis()          # chạy thử không cần server; thật: import redis; r = redis.Redis(host="localhost", port=6379)

calls = 0
def model_predict(text):            # thay cho model thật, đếm số lần bị gọi
    global calls; calls += 1
    return {"label": "tích cực", "len": len(text)}

def cached_predict(text, ttl=3600):
    key = "pred:v3:" + hashlib.sha256(text.encode()).hexdigest()   # v3: phiên bản model/prompt
    if (hit := r.get(key)) is not None:
        return json.loads(hit)
    out = model_predict(text)
    r.setex(key, ttl, json.dumps(out))
    return out

for q in ["ghế đẹp", "ghế đẹp", "giao chậm", "ghế đẹp"]:
    cached_predict(q)
print("4 yêu cầu, model chỉ chạy", calls, "lần; TTL còn lại:", r.ttl("pred:v3:" + hashlib.sha256("ghế đẹp".encode()).hexdigest()), "giây")

4 yêu cầu, model chỉ chạy 2 lần; TTL còn lại: 3599 giây


## Bài tập 4: Truy vấn cho bộ eval {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
# Cần Postgres có bảng conversations, messages của hệ thống bạn
import json, psycopg

SQL = """
WITH ranked AS (
  SELECT c.customer_id, m.content AS question, length(m.content) AS len,
         ROW_NUMBER() OVER (PARTITION BY c.customer_id ORDER BY length(m.content) DESC) AS rn
  FROM conversations c
  JOIN messages m ON m.conversation_id = c.id AND m.role = 'user'
  WHERE c.created_at >= date_trunc('month', now()) - interval '1 month'
    AND c.created_at <  date_trunc('month', now())
    AND (c.feedback = 'dislike' OR c.escalated)
)
SELECT customer_id, question FROM ranked WHERE rn <= 2 ORDER BY len DESC LIMIT 200;
"""

def export_eval(dsn, path="eval_candidates.jsonl"):
    with psycopg.connect(dsn) as conn, open(path, "w", encoding="utf-8") as f:
        for customer_id, question in conn.execute(SQL):
            f.write(json.dumps({"customer_id": customer_id, "question": question}, ensure_ascii=False) + "\n")

## Bài tập 5: SQL cho nhật ký chatbot {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import sqlite3
con = sqlite3.connect(":memory:")
con.executescript("""
CREATE TABLE conversations (id INTEGER PRIMARY KEY, user_id TEXT, model_version TEXT, created_at TEXT);
CREATE TABLE feedback (conv_id INTEGER, rating INTEGER);
INSERT INTO conversations VALUES (1,'u1','v1','2026-11-01'),(2,'u2','v1','2026-11-01'),(3,'u1','v2','2026-11-02'),
  (4,'u3','v2','2026-11-02'),(5,'u4','v2','2026-11-03'),(6,'u2','v1','2026-11-03'),(7,'u5','v2','2026-11-03');
INSERT INTO feedback VALUES (1,1),(2,0),(3,1),(4,1),(5,0),(7,1);
""")
# TODO: sql_theo_ban (version, số hội thoại có đánh giá, tỉ lệ hữu ích), sql_chua_danh_gia

In [ ]:
sql_theo_ban = """
SELECT c.model_version, COUNT(*) AS so_hoi_thoai, AVG(f.rating) AS ti_le_huu_ich
FROM conversations c JOIN feedback f ON f.conv_id = c.id
GROUP BY c.model_version ORDER BY c.model_version"""
sql_chua_danh_gia = """
SELECT COUNT(*) FROM conversations c LEFT JOIN feedback f ON f.conv_id = c.id WHERE f.conv_id IS NULL"""
for row in con.execute(sql_theo_ban):
    print(row)
print("chưa đánh giá:", con.execute(sql_chua_danh_gia).fetchone()[0])

('v1', 2, 0.5)
('v2', 4, 0.75)
chưa đánh giá: 1


## Bài tập 6: Tìm kiếm có phân quyền {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import sqlite3
con = sqlite3.connect(":memory:")
con.execute("CREATE TABLE docs (id INTEGER, phong_ban TEXT, noi_dung TEXT)")
con.executemany("INSERT INTO docs VALUES (?,?,?)", [
    (1, "nhan_su", "Bảng lương và thưởng năm 2026"), (2, "nhan_su", "Quy trình xét thưởng cuối năm"),
    (3, "chung", "Lịch nghỉ lễ và thưởng Tết"), (4, "kinh_doanh", "Chỉ tiêu doanh số và thưởng quý"),
    (5, "chung", "Hướng dẫn xin nghỉ phép"), (6, "kinh_doanh", "Chính sách thưởng cho đại lý")])

def tim(tu_khoa, nhom, k=2):
    # TODO: một câu SQL có điều kiện quyền + LIKE + LIMIT
    return []

def tim_sai(tu_khoa, nhom, k=2):
    rows = con.execute("SELECT id, phong_ban FROM docs WHERE noi_dung LIKE ? LIMIT ?", (f"%{tu_khoa}%", k)).fetchall()
    return [r for r in rows if r[1] in nhom or r[1] == "chung"]

In [ ]:
def tim(tu_khoa, nhom, k=2):
    dau = ",".join("?" * len(nhom))
    sql = f"SELECT id, phong_ban FROM docs WHERE (phong_ban IN ({dau}) OR phong_ban = 'chung') AND noi_dung LIKE ? LIMIT ?"
    return con.execute(sql, (*nhom, f"%{tu_khoa}%", k)).fetchall()

print("đúng:", tim("thưởng", ["kinh_doanh"]))
print("sai :", tim_sai("thưởng", ["kinh_doanh"]))

đúng: [(3, 'chung'), (4, 'kinh_doanh')]
sai : []


## Bài tập 7: Cache có thời hạn {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
class TTLCache:
    def __init__(self):
        self.data = {}
    def get(self, k, now):
        # TODO: trả về giá trị nếu còn hạn, ngược lại None (và xóa khóa hết hạn)
        return None
    def set(self, k, v, ttl, now):
        # TODO
        pass

log = [(0, "giờ mở cửa"), (5, "phí ship"), (20, "giờ mở cửa"), (40, "phí ship"), (65, "giờ mở cửa"),
       (70, "giờ mở cửa"), (90, "đổi trả"), (100, "phí ship"), (110, "đổi trả"), (130, "giờ mở cửa")]

In [ ]:
class TTLCache:
    def __init__(self):
        self.data = {}
    def get(self, k, now):
        v = self.data.get(k)
        if v is None or now >= v[1]:
            self.data.pop(k, None)
            return None
        return v[0]
    def set(self, k, v, ttl, now):
        self.data[k] = (v, now + ttl)

cache, trung = TTLCache(), 0
for t, q in log:
    if cache.get(q, t) is not None:
        trung += 1; nguon = "cache"
    else:
        cache.set(q, f"trả lời cho '{q}'", 60, t); nguon = "model"
    print(f"t={t:3d}s {q:12s} → {nguon}")
print(f"trúng {trung}/{len(log)}")

t=  0s giờ mở cửa   → model
t=  5s phí ship     → model
t= 20s giờ mở cửa   → cache
t= 40s phí ship     → cache
t= 65s giờ mở cửa   → model
t= 70s giờ mở cửa   → cache
t= 90s đổi trả      → model
t=100s phí ship     → model
t=110s đổi trả      → cache
t=130s giờ mở cửa   → model
trúng 4/10


## Bài tập 8: Hàng đợi tác vụ có thử lại và hàng đợi lỗi {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
from collections import deque
tac_vu = [f"doc_{i}.pdf" for i in range(7)] + ["hong.pdf"]
LOI_TAM = {"doc_2.pdf": 1, "doc_5.pdf": 2}            # số lần thất bại trước khi thành công

def xu_ly(ten, lan_thu):
    if ten == "hong.pdf":
        raise ValueError("file hỏng, không đọc được")
    if lan_thu <= LOI_TAM.get(ten, 0):
        raise TimeoutError("dịch vụ embedding quá tải")
    return f"đã nạp {ten}"
# TODO: hang = deque((t, 1) for t in tac_vu); vòng worker; thanh_cong, dead_letter, so_lan_xu_ly

In [ ]:
hang = deque((t, 1) for t in tac_vu)
thanh_cong, dead_letter, so_lan_xu_ly = [], [], 0
while hang:
    ten, lan = hang.popleft()
    so_lan_xu_ly += 1
    try:
        thanh_cong.append(xu_ly(ten, lan))
    except Exception as e:
        if lan < 3:
            hang.append((ten, lan + 1))
        else:
            dead_letter.append((ten, f"{type(e).__name__}: {e}"))
print(f"thành công {len(thanh_cong)}, dead-letter {dead_letter}, tổng {so_lan_xu_ly} lần xử lý")

thành công 7, dead-letter [('hong.pdf', 'ValueError: file hỏng, không đọc được')], tổng 13 lần xử lý
